# Validating data with Pandera

## Chosen topic
Data validation with a focus on Pandera, applied to email campaign data.

The goal is to learn how to define and use schemas in Pandera, show which rows pass and which are rejected, as well as why. 

Reliable data is a fundamental prerequisite for analyses and predictive models, which connects directly to my internship where data quality is part of the work.


## Definition

Pandera is an open source project that allows us to perform data validation on dataframe-like objects.
The goal is to make data processing pipelines more readable and robust. 

Pandera makes it possible to:  
- Define a schema once and use it to validate different dataframe types.  
- Check the types and properties of columns in a pd.DataFrame or values in a pd.Series.
- Perform more complex statistical validation like hypothesis testing.
- Parse data to standardize the preprocessing steps needed to produce valid data.
- Integrate with existing data analysis/processing pipelines viafunction decorators.  
- Define dataframe models.  
- Synthesize data from schema objects for property-based testing with pandas data structures.  
- Lazily validate dataframes so that all validation rules are executed before raising an error.  
- Integrate with a rich ecosystem of python tools like pydantic and fastapi.


## Starting questions
1. How do I define a schema and what does it check?
2. What constraints can I put on columns beyond dtype? 
3. What happens when validation fails?
4. How does Pandera fit into a pipeline?
5. What are the limitations with real messy data?

In [1]:
from importlib.metadata import version
import pandas as pd
import pandera.pandas as pa # importing like this follows the current best practices for pandas-specific validation.
from pandera.pandas import Column, DataFrameSchema, Check

print(version("pandera")) # latest stable version 0.33.1 released 1 sept 2026

0.33.1


Sources:
Pandera documentation: https://pandera.readthedocs.io/en/stable/

A schema in Pandera is like a contract that describes the expected structure and properties of the data. It is like a blueprint that specifies data types, acceptable value ranges and relationships between columns.
When validating a df against a schema, Pandera checks that every aspect matches the specifications.

Schemas consist in column definition, each with data type and optional constraints.

Pandera offers two main approaches: 
- Object-based API
- Class-based API

Validation checks three key aspects:
- Structure (are all the required columns there?)
- Types (is each column the correct datatype?)
- Values (do values satisfy all the constraints?)

Source: 
Statology: https://www.statology.org/data-validation-in-python-with-pandera-a-practical-introduction/

In [2]:
# Creating basic email campaign data
data = {
    'subject_line': ['Spring Sale', 'Newsletter #42', 'Spring Sale', 'Product Launch', 'Newsletter #42'],
    'recipients': [1200, 1150, 1200, 1300, 1150],
    'sent_at': ['2026-03-01', '2026-03-15', '2026-03-01', '2026-04-02', '2026-03-15'],
    'opens': [340, 210, 340, 520, 190],
    'cost_per_email': [0.05, 0.08, 0.05, 0.12, 0.08],
}

df = pd.DataFrame(data)
df['sent_at'] = pd.to_datetime(df['sent_at'])

### Basic type validation

In [3]:
# Creating a schema that validates structure and types
schema = DataFrameSchema({
    'subject_line': Column(str),
    'recipients': Column(int, Check.greater_than(0)),
    'sent_at': Column(pa.DateTime),
    'opens': Column(int, Check.greater_than_or_equal_to(0)),
    'cost_per_email': Column(float, Check.greater_than(0))
})

# Valideting the df
validated_df = schema.validate(df)

There is no Column(datetime) so to validate the date column pandera needs pandas dtype string specification Column('datetime64[ns]') or pandera's alias Column(pa.DateTime).

There is no output because the method returns the df silently when it succeeds.  
It raises an exception when it fails.

### Checks and categorical contraints

In [4]:
enhanced_schema = DataFrameSchema({
    'subject_line': Column(str, Check.isin(['Spring Sale', 'Newsletter #42', 'Product Launch'])),
    'recipients': Column(int, Check.in_range(1, 5000)),
    'sent_at': Column(pa.DateTime, Check.greater_than_or_equal_to(pd.Timestamp('2026-01-01'))),
    'opens': Column(int, Check.in_range(0, 5000)),
    'cost_per_email': Column(float, Check.in_range(0, 0.2))
})

validated_df = enhanced_schema.validate(df)

### Cross-Column Validation

In [5]:
# Certain column combinations should make sense: opens <= recipients

def check_opens_vs_recipients(df):
    return (df['opens'] <= df['recipients']).all()

full_schema = DataFrameSchema({
    'subject_line': Column(str, Check.isin(['Spring Sale', 'Newsletter #42', 'Product Launch'])),
    'recipients': Column(int, Check.in_range(1, 5000)),
    'sent_at': Column(pa.DateTime, Check.greater_than_or_equal_to(pd.Timestamp('2026-01-01'))),
    'opens': Column(int, Check.in_range(0, 5000)),
    'cost_per_email': Column(float, Check.in_range(0, 0.2))
}, 
checks=Check(check_opens_vs_recipients, error="The openings should be equal or less than the recipients."))

validated_df = full_schema.validate(df)

### Validation failure

In [6]:
# Creating bad data
bad_data = {
    'subject_line': ['Spring Sale', 'Newsletter', 'Spring Sale', 'Product Launch', 'Newsletter #42'],
    'recipients': [1200, 1150, 1200, 1300, 1150],
    'sent_at': ['2027-03-01', '2026-03-15', '2026-03-01', '2026-04-02', '2026-03-15'],
    'opens': [1220, 210, 340, 520, 190],
    'cost_per_email': [0.05, 0.08, 0.05, 0.12, 3],
}

bad_df = pd.DataFrame(bad_data)
bad_df['sent_at'] = pd.to_datetime(bad_df['sent_at'])

In [ ]:
# Running the bad_data df through the full schema to analyze the error

# validate_bad_data = full_schema.validate(bad_df, lazy=True)

# Raises SchemaError and stops. Also shows the checks that were broken.

SchemaErrors: {
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": null,
                "column": "subject_line",
                "check": "isin(['Spring Sale', 'Newsletter #42', 'Product Launch'])",
                "error": "Column 'subject_line' failed element-wise validator number 0: isin(['Spring Sale', 'Newsletter #42', 'Product Launch']) failure cases: Newsletter"
            },
            {
                "schema": null,
                "column": "cost_per_email",
                "check": "in_range(0, 0.2)",
                "error": "Column 'cost_per_email' failed element-wise validator number 0: in_range(0, 0.2) failure cases: 3.0"
            },
            {
                "schema": null,
                "column": null,
                "check": "The openings should be equal or less than the recipients.",
                "error": "DataFrameSchema 'None' failed series or dataframe validator 0: <Check check_opens_vs_recipients: The openings sh

### Handling validation error gracefully

In production pipelines, validation errors need to be handled without stopping the execution. And Pandera provides detailes error information that can be used for logging, data cleaning, user feedback.